In [1]:
# ==============================================================================
# 1. CÀI ĐẶT THƯ VIỆN
# ==============================================================================
print("⏳ Đang cài đặt thư viện FinRL (vui lòng chờ 1-2 phút)...")
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git -q
!pip install shimmy>=0.2.1 -q

⏳ Đang cài đặt thư viện FinRL (vui lòng chờ 1-2 phút)...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.7/108.7 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.9/84.9 kB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 54.8 MB/s

In [2]:
# ==============================================================================
# 2. KHỞI TẠO HỆ THỐNG
# ==============================================================================
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# Import FinRL
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl import config
from finrl.config import INDICATORS

# Import Stable-Baselines3 (Full Algorithms)
from stable_baselines3 import DDPG, TD3, SAC, A2C, PPO
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import ProgressBarCallback

# Tạo thư mục lưu trữ chung
if not os.path.exists("./" + config.TRAINED_MODEL_DIR):
    os.makedirs("./" + config.TRAINED_MODEL_DIR)

print("\n✅ CÀI ĐẶT HOÀN TẤT! Sẵn sàng làm việc.")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=


✅ CÀI ĐẶT HOÀN TẤT! Sẵn sàng làm việc.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
# ==============================================================================
# 2. CHUẨN BỊ DỮ LIỆU & MÔI TRƯỜNG (SAFE MODE)
# ==============================================================================

# 1. Load dữ liệu
if not os.path.exists('train_data.csv'):
    print("❌ LỖI: Chưa có file 'train_data.csv'. Hãy upload lên Colab!")
else:
    train = pd.read_csv('train_data.csv')
    train = train.sort_values(['date', 'tic']).reset_index(drop=True)
    train.index = train.date.factorize()[0]
    print(f"📊 Dữ liệu đã load: {train.shape}")

    # 2. Cấu hình môi trường
    stock_dimension = len(train.tic.unique())
    state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension

    env_kwargs = {
        "hmax": 100, "initial_amount": 1000000,
        "num_stock_shares": [0]*stock_dimension,
        "buy_cost_pct": [0.001]*stock_dimension, "sell_cost_pct": [0.001]*stock_dimension,
        "state_space": state_space, "stock_dim": stock_dimension,
        "tech_indicator_list": INDICATORS, "action_space": stock_dimension, "reward_scaling": 1e-4
    }

    # 3. Tạo môi trường Gym & Bọc Safe Mode (DummyVecEnv)
    e_train_gym = StockTradingEnv(df = train, **env_kwargs)
    env_train_sb3 = DummyVecEnv([lambda: e_train_gym])

    # 4. Tạo Noise (Dùng cho DDPG/TD3)
    n_actions = e_train_gym.action_space.shape[-1]
    action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

    print("✅ MÔI TRƯỜNG SAFE MODE ĐÃ SẴN SÀNG!")

📊 Dữ liệu đã load: (153356, 19)
✅ MÔI TRƯỜNG SAFE MODE ĐÃ SẴN SÀNG!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
# ==============================================================================
# 3. HUẤN LUYỆN MODEL: A2C
# ==============================================================================
print("\n👉 [1/5] Đang train A2C...")

# Tạo thư mục
if not os.path.exists("trained_models/a2c"): os.makedirs("trained_models/a2c")

# Khởi tạo và Train
model_a2c = A2C("MlpPolicy", env_train_sb3, verbose=0, learning_rate=0.0007, ent_coef=0.01)
model_a2c.learn(total_timesteps=30000, callback=ProgressBarCallback())

# Lưu
model_a2c.save("trained_models/a2c/agent_a2c")
print("✅ A2C Hoàn tất.")


👉 [1/5] Đang train A2C...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

...

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

✅ A2C Hoàn tất.


In [5]:
# ==============================================================================
# 4. HUẤN LUYỆN MODEL: PPO
# ==============================================================================
print("\n👉 [2/5] Đang train PPO...")

if not os.path.exists("trained_models/ppo"): os.makedirs("trained_models/ppo")

model_ppo = PPO("MlpPolicy", env_train_sb3, verbose=0, learning_rate=0.00025, n_steps=2048, ent_coef=0.01)
model_ppo.learn(total_timesteps=30000, callback=ProgressBarCallback())

model_ppo.save("trained_models/ppo/agent_ppo")
print("✅ PPO Hoàn tất.")

Output()


👉 [2/5] Đang train PPO...


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

...

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

✅ PPO Hoàn tất.


In [6]:
# ==============================================================================
# 5. HUẤN LUYỆN MODEL: DDPG
# ==============================================================================
print("\n👉 [3/5] Đang train DDPG...")

if not os.path.exists("trained_models/ddpg"): os.makedirs("trained_models/ddpg")

model_ddpg = DDPG("MlpPolicy", env_train_sb3, action_noise=action_noise,
                  batch_size=128, buffer_size=50000, learning_rate=0.001, verbose=0)
model_ddpg.learn(total_timesteps=30000, callback=ProgressBarCallback())

model_ddpg.save("trained_models/ddpg/agent_ddpg")
print("✅ DDPG Hoàn tất.")

Output()


👉 [3/5] Đang train DDPG...


/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (

...

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

✅ DDPG Hoàn tất.


In [7]:
# ==============================================================================
# 6. HUẤN LUYỆN MODEL: TD3
# ==============================================================================
print("\n👉 [4/5] Đang train TD3...")

if not os.path.exists("trained_models/td3"): os.makedirs("trained_models/td3")

model_td3 = TD3("MlpPolicy", env_train_sb3, action_noise=action_noise,
                batch_size=128, buffer_size=50000, learning_rate=0.001, verbose=0)
model_td3.learn(total_timesteps=30000, callback=ProgressBarCallback())

model_td3.save("trained_models/td3/agent_td3")
print("✅ TD3 Hoàn tất.")

Output()


👉 [4/5] Đang train TD3...


/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (

...

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (

✅ TD3 Hoàn tất.


In [8]:
# ==============================================================================
# 7. HUẤN LUYỆN MODEL: SAC
# ==============================================================================
print("\n👉 [5/5] Đang train SAC...")

if not os.path.exists("trained_models/sac"): os.makedirs("trained_models/sac")

model_sac = SAC("MlpPolicy", env_train_sb3, batch_size=128, buffer_size=50000,
                learning_rate=0.0001, ent_coef='auto', verbose=0)
model_sac.learn(total_timesteps=30000, callback=ProgressBarCallback())

model_sac.save("trained_models/sac/agent_sac")
print("✅ SAC Hoàn tất.")

Output()


👉 [5/5] Đang train SAC...


/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (

...

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/finrl/meta/env_stock_trading/env_stocktrading.py:189: RuntimeWarning: 
divide by zero encountered in scalar floor_divide
  available_amount = self.state[0] // (

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

✅ SAC Hoàn tất.


In [9]:
# ==============================================================================
# 8. TỔNG KẾT & TẢI VỀ
# ==============================================================================
import os

print("🎉 CHÚC MỪNG! BẠN ĐÃ TRAIN XONG CẢ 5 MODEL.")
print("📦 Đang nén dữ liệu...")

!zip -r all_5_models_safe.zip trained_models/

print("\n👉 File 'all_5_models_safe.zip' đã sẵn sàng. Hãy tải về máy!")

🎉 CHÚC MỪNG! BẠN ĐÃ TRAIN XONG CẢ 5 MODEL.
📦 Đang nén dữ liệu...
  adding: trained_models/ (stored 0%)
  adding: trained_models/sac/ (stored 0%)
  adding: trained_models/sac/agent_sac.zip (stored 0%)
  adding: trained_models/a2c/ (stored 0%)
  adding: trained_models/a2c/agent_a2c.zip (stored 0%)
  adding: trained_models/ddpg/ (stored 0%)
  adding: trained_models/ddpg/agent_ddpg.zip (stored 0%)
  adding: trained_models/td3/ (stored 0%)
  adding: trained_models/td3/agent_td3.zip (stored 0%)
  adding: trained_models/ppo/ (stored 0%)
  adding: trained_models/ppo/agent_ppo.zip (stored 0%)

👉 File 'all_5_models_safe.zip' đã sẵn sàng. Hãy tải về máy!
